In [1]:
import pandas as pd
import re
import unicodedata
import glob
import os
from config_schema import SURVEY_SCHEMA, CNAE_MAP, QUEST_MAPPING, KEYWORD_RULES, SCORING_MAPS

def read_parse_csv(file_path: str) -> pd.DataFrame:
    """
    CSV reader with multiple encoding support.
    """
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding, sep=';')
        except UnicodeDecodeError:
            continue
        except Exception:
            continue
    
    print(f"WARNING: Could not read file {file_path}")
    return pd.DataFrame()

def normalize_questions_id(text: str) -> str:
    """
    Standardizes question string to create a unique key.
    """
    if pd.isna(text) or re.match(r'^\d', str(text)):
        return None
    
    text = str(text).lower().strip()
    text = re.sub(r'\s*\(.*?\)', '', text)
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode('ascii')
    text = text.replace('/', ' ').replace('-', ' ').replace('.', ' ')
    text = re.sub(r'"', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', '_', text)
    return text[:50]

def normalize_free_text(text: str, rule_type: str) -> str:
    """
    Basing on question value type access to keywords dictionary and categorize them.
    """
    if not text or str(text).lower() == 'nan':
        return "No especificado"
        
    clean_text = str(text).lower().strip()
    rules = KEYWORD_RULES.get(rule_type, {})

    for category, keywords in rules.items():
        for keyword in keywords:
            if keyword in clean_text:
                return category
                
    return str(text).title()

def normalize_response_value(row: pd.Series) -> str | int | float:
    """
    Normalize raw CSV answers into clean Data.
    """
    raw_val = str(row['Valor']).strip()
    field_id = str(row.get('internal_id', ''))
    
    schema_options = SURVEY_SCHEMA.get(field_id, [])
    if schema_options == ['NUMERIC']:
        digits = re.sub(r'[^\d\.,]', '', raw_val).replace(',', '.')
        try:
            # If it has decimal, return float else int
            return float(digits) if '.' in digits else int(digits)
        except ValueError:
            return 0

    if "cnae" in field_id or "profile" in field_id:
        if raw_val.isdigit(): 
            return CNAE_MAP.get(raw_val, "Otro")
        return normalize_free_text(raw_val, rule_type="sector")

    if field_id in ["erp_in_use", "crm_in_use", "powerbi_usage"]:
        # Check if value is in predefined csv questions options if not use keyword dictionary
        if raw_val in schema_options: 
            return raw_val
        return normalize_free_text(raw_val, rule_type="software")

    if "channel" in field_id or "comunicacion" in field_id:
        return normalize_free_text(raw_val, rule_type="channel")

    if "antivirus" in field_id:
        return normalize_free_text(raw_val, rule_type="antivirus")


    clean_val = raw_val.replace('"', '') 
    clean_val = re.sub(r'([Mm]enos (de|del)|[Mm]enor que)\s+', '<', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'([Mm]ás (de|del)|[Mm]ayor que)\s+', '>', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'Entre\s+(.*?)\s+y\s+(.*)', r'\1-\2', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'\s*-\s*', '-', clean_val)

    options = schema_options
    if not options or options == ["TEXT"]:
        return clean_val.title()

    for opt in options:
        if clean_val.lower() == opt.lower():
            return opt 
        opt_clean = opt.split('(')[0].strip()
        if len(opt_clean) > 3 and opt_clean.lower() in clean_val.lower():
            return opt
            
    return clean_val


def process_surveys(data_folder: str) -> pd.DataFrame:
    """
    Loops through CSVs, merges with schema, cleans data and unifies client data.
    """
    all_data = []
    files = glob.glob(os.path.join(data_folder, '*.csv'))
    print(f"Found {len(files)} files in {data_folder}")

    for file_path in files:            
        try:
            answers_df = read_parse_csv(file_path)
            if answers_df.empty or 'Campo' not in answers_df.columns:
                continue

            answers_df['questions'] = answers_df['Campo'].apply(normalize_questions_id)
            answers_df = answers_df.dropna(subset=['questions'])
            answers_df['internal_id'] = answers_df['questions'].map(QUEST_MAPPING).fillna(answers_df['questions'])
            answers_df['normalized_value'] = answers_df.apply(normalize_response_value, axis=1)

            # Create a single row dataframe for this client
            client_row = answers_df[['internal_id', 'normalized_value']].set_index('internal_id').T
            
            # Add metadata
            client_row.insert(0, 'source_file', os.path.basename(file_path))
            
            all_data.append(client_row)

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()


DATA_DIR = './data/'
SCHEMA_FILE = './data/questions.csv'

print("Processing Survey Files...")
master_df = process_surveys(DATA_DIR)
    
if not master_df.empty:
    print("-" * 30)
    print(f"SUCCESS: Aggregated {len(master_df)} surveys.")
    print("-" * 30)
    print(master_df.head())
    
else:        
    print("No valid data was generated.")

Processing Survey Files...
Found 22 files in ./data/
------------------------------
SUCCESS: Aggregated 21 surveys.
------------------------------
internal_id                             source_file company_profile_cnae  \
0                                        answer.csv           Tecnología   
1               form_data_69666a746084f_cealvet.csv             Comercio   
2            form_data_693941bba67fc_siesystems.csv           Industrial   
3               form_data_6957ae9fd6241_sorolla.csv             Comercio   
4                                       answer2.csv            Servicios   

internal_id company_postcode number_of_employees average_employee_age  \
0                      28023                  50                30-40   
1                      43500                   7                30-40   
2                      26006                  12                30-40   
3                      43500                  19                41-50   
4                      08021   

In [2]:
def calculate_maturity_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates Digital Maturity Scores (0-100) 
    """
    
    def get_score(col_name, map_type):
        """
        Retrieves score using SCORING_MAPS, score rules predefined in config_schema.
        """
        if col_name not in df.columns:
            return 0
        
        # Select scoring maps from config schema
        mapping_dict = SCORING_MAPS.get(map_type, {})

        return df[col_name].map(mapping_dict).fillna(0)

    # Weights: Processes (40%), Infrastructure (30%), Collaboration (30%)
    df['KPI_OPERATIONS'] = (
        get_score('key_processes_digitized_pct', 'percentage') * 0.4 + 
        get_score('it_infrastructure_type', 'infrastructure') * 0.3 +
        get_score('collaboration_tools_usage', 'binary') * 0.3 
    )

    # Focus: Is tech actually making money?
    # Weights: Revenue (40%), AI (30%), Web Presence (30%)
    df['KPI_BUSINESS'] = (
        get_score('digital_revenue', 'revenue') * 0.4 +
        get_score('ai_for_automation_usage', 'ai') * 0.3 +
        get_score('active_internet_presence', 'binary') * 0.3
    )


    # Focus: Risk management, 25% each one
    df['KPI_SECURITY'] = (
        get_score('two_factor_authentication', 'binary') * 0.25 +
        get_score('continuity_and_recovery_plans', 'binary') * 0.25 +
        get_score('phishing_simulations', 'binary') * 0.25 +
        get_score('data_protection_compliance', 'binary') * 0.25
    )
    
    # Focus: Human capital
    # Weights: Advanced Skills (50%), Cybersecurity Training (50%)
    df['KPI_CULTURE'] = (
        get_score('advanced_digital_skills_pct', 'percentage') * 0.5 +
        get_score('cybersecurity_training', 'binary') * 0.5
    )

    # Calculate dmi global score
    df['GLOBAL_SCORE'] = (
        df['KPI_OPERATIONS'] * 0.30 +
        df['KPI_SECURITY'] * 0.30 +
        df['KPI_BUSINESS'] * 0.20 +
        df['KPI_CULTURE'] * 0.20
    ).round(1)

    # Assign maturity labels based on global score
    def assign_label(score):
        if score >= 80: 
            return 'Líder Digital'
        if score >= 60: 
            return 'Avanzado'
        if score >= 40: 
            return 'En Desarrollo'
        return 'Principiante Digital'

    df['MATURITY_LABEL'] = df['GLOBAL_SCORE'].apply(assign_label)
    return df

master_df = calculate_maturity_kpis(master_df)

In [3]:
import numpy as np
import json

def generate_benchmark_reference(df: pd.DataFrame) -> dict:
    """
    Takes the master dataframe with dmi calculated and aggregates it into 
    a json dictionary for benchmarking.
    """
    
    # Establish company size categories
    conditions = [
        (df['number_of_employees'] <= 10),
        (df['number_of_employees'] > 10) & (df['number_of_employees'] <= 50),
        (df['number_of_employees'] > 50) & (df['number_of_employees'] <= 250),
        (df['number_of_employees'] > 250)
    ]
    choices = ['Micro', 'Small', 'Medium', 'Large']
    df['company_size'] = np.select(conditions, choices, default='Unknown')

    score_cols = [
        'KPI_OPERATIONS',
        'KPI_SECURITY',
        'KPI_BUSINESS', 
        'KPI_CULTURE',
        'GLOBAL_SCORE',
    ]
    
    # We check the % of companies that have these implemented (yes/no)
    adoption_cols = [
        'two_factor_authentication',
        'continuity_and_recovery_plans',
        'microsoft_365_usage',
        'remote_work_acceptable_use_policy',
        'phishing_simulations',
        'active_internet_presence',
        'data_protection_compliance',
        'regular_patching_and_updates',
        'incident_response_plan',
        'digital_marketing_use',
        'accessible_digital_sales_channels',
        'continuous_digital_training',
        'ai_for_automation_usage'
    ]
    
    # We check the most popular tools in each sector(top 3)
    market_cols = [
        'erp_in_use', 
        'crm_in_use', 
        'it_infrastructure_type',
        'antivirus_used',
        'powerbi_usage',
        'priority_assessment_area',
        'average_employee_age',
        'it_outsourcing_level',
        'digital_revenue'     
    ]

    outsourcing_map = {'Bajo': 1, 'Medio': 2, 'Alto': 3}

    reference_data = {}
    # Group by sector + size ("Industrial" + "Small")
    grouped = df.groupby(['company_profile_cnae', 'company_size'])
    
    print(f"Generating benchmarks for {len(grouped)}...")

    for (sector, size), group in grouped:
        # Unique ID ("Industrial_Small")
        combination_id = f"{sector}_{size}"
        
        # Minimum 3 samples to consider valid benchmark
        if len(group) < 3: 
            continue
            
        stats = {
            "meta": {
                "sector": sector,
                "size": size,
                "sample_size": len(group)
            },
            "scores": {},
            "adoption_rates": {},
            "market_leaders": {},
            "averages": {}
        }
        
        for col in score_cols:
            if col in group.columns:
                stats["scores"][col] = {
                    "p25": float(round(group[col].quantile(0.25), 1)),
                    "median": float(round(group[col].median(), 1)),
                    "p75": float(round(group[col].quantile(0.75), 1))
                }
        
        # Percentage of "Yes" answers
        for col in adoption_cols:
            if col in group.columns:
                is_yes = group[col].astype(str).str.contains(r'Sí', case=False, regex=True)
                adoption_pct = is_yes.mean()
                stats["adoption_rates"][col] = float(round(adoption_pct * 100, 1))

        # Calculate top 3 tools can involve ranges questions
        for col in market_cols:
            if col in group.columns:
                counts = group[col].value_counts(normalize=True).head(3)
                
                top_list = []
                for tool_name, share in counts.items():
                    top_list.append({
                        "tool": tool_name,
                        "share_pct": float(round(share * 100, 1))
                    })

                if col == 'priority_assessment_area':
                    stats["sector_priorities"] = top_list
                else:
                    stats["market_leaders"][col] = top_list

        if 'it_outsourcing_level' in group.columns:
            numeric_vals = group['it_outsourcing_level'].map(outsourcing_map).dropna()
            if not numeric_vals.empty:
                avg_val = numeric_vals.mean()
                # Convert back to text for display (1=Bajo, 2=Medio, 3=Alto)
                label = "Bajo" if avg_val < 1.5 else "Alto" if avg_val > 2.5 else "Medio"
                stats["averages"]['it_outsourcing_level'] = {
                    "avg_score": float(round(avg_val, 2)),
                    "label": label
                }

        reference_data[combination_id] = stats

    return reference_data


benchmark_atlas = generate_benchmark_reference(master_df)

with open('benchmark_data.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark_atlas, f, indent=4, ensure_ascii=False)    
    print("Success! 'benchmark_data.json' has been created.")
    
    example_keys = list(benchmark_atlas.keys())
    if example_keys:
        first_key = example_keys[0]
        print(f"\n--- Preview: {first_key} ---")
        print(json.dumps(benchmark_atlas[first_key], indent=2, ensure_ascii=False))

Generating benchmarks for 10...
Success! 'benchmark_data.json' has been created.

--- Preview: Comercio_Micro ---
{
  "meta": {
    "sector": "Comercio",
    "size": "Micro",
    "sample_size": 4
  },
  "scores": {
    "KPI_OPERATIONS": {
      "p25": 77.0,
      "median": 89.5,
      "p75": 100.0
    },
    "KPI_SECURITY": {
      "p25": 25.0,
      "median": 43.8,
      "p75": 62.5
    },
    "KPI_BUSINESS": {
      "p25": 50.0,
      "median": 51.0,
      "p75": 54.6
    },
    "KPI_CULTURE": {
      "p25": 78.1,
      "median": 93.8,
      "p75": 100.0
    },
    "GLOBAL_SCORE": {
      "p25": 66.9,
      "median": 67.5,
      "p75": 67.6
    }
  },
  "adoption_rates": {
    "two_factor_authentication": 25.0,
    "continuity_and_recovery_plans": 25.0,
    "microsoft_365_usage": 100.0,
    "remote_work_acceptable_use_policy": 100.0,
    "phishing_simulations": 0.0,
    "active_internet_presence": 100.0,
    "data_protection_compliance": 100.0,
    "regular_patching_and_updates": 100

In [4]:
master_df.head(20).style

internal_id,source_file,company_profile_cnae,company_postcode,number_of_employees,average_employee_age,number_of_clients,number_of_suppliers,annual_revenue,it_outsourcing_level,remote_work_acceptable_use_policy,secure_remote_access,two_factor_authentication,it_infrastructure_type,key_processes_digitized_pct,erp_in_use,crm_in_use,ai_for_automation_usage,database_type,powerbi_usage,advanced_digital_skills_pct,microsoft_365_usage,collaboration_tools_usage,continuous_digital_training,cybersecurity_training,ftfe_training,phishing_simulations,active_internet_presence,active_social_media_management,digital_marketing_use,visitor_follower_analysis,accessible_digital_sales_channels,digital_revenue,usual_customer_communication_channel,preferred_customer_communication_channel,antivirus_used,employees_using_antivirus_pct,regular_patching_and_updates,network_controls_implemented,documented_account_lifecycle_process,clear_roles_and_privileges,incident_response_plan,continuity_and_recovery_plans,data_protection_compliance,legal_and_compliance_training,priority_assessment_area,KPI_OPERATIONS,KPI_BUSINESS,KPI_SECURITY,KPI_CULTURE,GLOBAL_SCORE,MATURITY_LABEL,company_size
0,answer.csv,Tecnología,28023,50,30-40,20,28,>20M,Alto,Sí,Sí,Sí,Híbrida,51-75%,A3 / Wolters Kluwer,A3 / Wolters Kluwer,En producción,OnPremise,Power BI,76-100%,Sí,Sí,Sí,<1 vez al año,Sí,Sí,Sí,Sí,Sí,Sí,En Desarrollo,30-60%,Teléfono,Email,Avast,99,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Gestión de incidencias y continuidad de negocio,81.000000,90.000000,100.000000,50.000000,82.300000,Líder Digital,Small
1,form_data_69666a746084f_cealvet.csv,Comercio,43500,7,30-40,51,16,1-5M,Bajo,Sí,Sí,Parcial,On-premise,76-100%,Facturascript,Facturascript,En piloto,OnPremise,Power BI,76-100%,Sí,Sí,Sí,<1 vez al año,Sí,No,Sí,Sí,No,Sí,No,<10%,Teléfono,Teléfono,Panda Security,100,Sí,Sí,No,Sí,No,Sí,Sí,No,Procesos y automatización,79.000000,62.500000,62.500000,50.000000,65.000000,Avanzado,Micro
2,form_data_693941bba67fc_siesystems.csv,Industrial,26006,12,30-40,25,5,<1M,Bajo,No,Sí,Sí,Híbrida,26-50%,A3 / Wolters Kluwer,Wolfcrm,Explorando,Cloud,Power BI,51-75%,Sí,Sí,Ocasional,<1 vez al año,Sí,Puntual,Sí,Sí,Sí,Sí,No,10-30%,WhatsApp,Email,Eset Nod32,100,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Infraestructuras y conectividad,71.000000,62.000000,75.000000,37.500000,63.700000,Avanzado,Small
3,form_data_6957ae9fd6241_sorolla.csv,Comercio,43500,19,41-50,1000,200,1-5M,Bajo,No,No,No,Híbrida,0-25%,Ninguno,Ninguno,No,Cloud,Ninguno,0-25%,Sí,Sí,No,No,No,No,Sí,No,No,No,Sí,<10%,Email,Email,Microsoft Defender,0,No,No,No,No,No,No,No,No,Ciberseguridad,61.000000,40.000000,0.000000,12.500000,28.800000,Principiante Digital,Small
4,answer2.csv,Servicios,08021,6,41-50,181,99,<1M,Alto,En desarrollo,Sí,Sí,Híbrida,0-25%,A3 / Wolters Kluwer,A3 / Wolters Kluwer,Explorando,OnPremise,Microsoft Excel,51-75%,Sí,Sí,Ocasional,No,No,No,Sí,No,Sí,Sí,No,<10%,Email,Email,Eset Nod32,100,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Procesos y automatización,61.000000,52.000000,75.000000,37.500000,58.700000,En Desarrollo,Micro
5,answer3.csv,Servicios,43500,25,41-50,1100,10,1-5M,Medio,Sí,Sí,Sí,Cloud,51-75%,A3 / Wolters Kluwer,Ninguno,En producción,Cloud,Ninguno,51-75%,Sí,Sí,Ocasional,<1 vez al año,Sí,No,Parcial,Ocasional,En evaluación,Parcial,No,<10%,Email,Email,Kaspersky,100,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Parcial,Procesos y automatización,90.000000,55.000000,75.000000,37.500000,68.000000,Avanzado,Small
6,form_data_6932c40287dd4.csv,Industrial,08292,30,30-40,1000,20,5-20M,Bajo,En desarrollo,Sí,Sí,Híbrida,26-50%,A3 / Wolters Kluwer,Wolfcrm,Explorando,Cloud,Power BI,26-50%,Sí,Sí,Ocasional,<1 vez al año,Sí,No,Sí,Sí,Sí,Sí,No,<10%,WhatsApp,Email,Eset Nod32,100,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Datos y analítica,71.000000,52.000000,75.000000,25.000000,59.200000,En Desarrollo,Small
7,form_data_693947c4ac3ea_puntdoc.csv,Servicios,08960,5,41-50,200,50,1-5M,Medio,En desarrollo,Sí,Parcial,Híbrida,26-50%,A3 / Wolters Kluwer,Odoo,Explorando,OnPremise,A3 / Wolters Kluwer,76-100%,Sí,Sí,Ocasional,No,Sí,No,Sí,No,No,No,No,<10%,Email,Emai

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import base64
from io import BytesIO
from jinja2 import Template
import datetime

def plot_to_base64(fig):
    """Converts plot to base64 string for HTML embedding"""
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return img_str

def create_executive_score_dist(df):
    """Distribution of Global Scores"""
    plt.figure(figsize=(8, 4))
    sns.histplot(df['GLOBAL_SCORE'], bins=20, kde=True, color='#2ecc71')
    plt.axvline(df['GLOBAL_SCORE'].mean(), color='red', linestyle='--', label='Market Avg')
    plt.title("Market Maturity Distribution")
    plt.xlabel("Score (0-100)")
    plt.legend()
    return plot_to_base64(plt.gcf())

def create_sector_comparison(df):
    """Sector comparison Box Plot"""
    plt.figure(figsize=(10, 5))
    order = df.groupby('company_profile_cnae')['GLOBAL_SCORE'].median().sort_values(ascending=False).index
    sns.boxplot(x='GLOBAL_SCORE', y='company_profile_cnae', data=df, order=order, palette="Blues_r")
    plt.title("Digital Maturity by Sector")
    plt.xlabel("Score")
    plt.ylabel("")
    return plot_to_base64(plt.gcf())

def create_risk_matrix(df):
    """Security Adoption Bar Chart"""
    risk_cols = [
        'two_factor_authentication',
        'continuity_and_recovery_plans',
        'regular_patching_and_updates',
        'phishing_simulations'
    ]
    # Hardcoded pretty labels
    risk_labels = {
        'two_factor_authentication': 'Doble Factor',
        'continuity_and_recovery_plans': 'Plan Continuidad',
        'regular_patching_and_updates': 'Actualizaciones',
        'phishing_simulations': 'Simulacros Phishing'
    }
    # Calculate % Yes
    adoption = {}
    for col in risk_cols:
        if col in df.columns:
            pct = df[col].astype(str).str.contains(r'si|sí|yes', case=False).mean() * 100
            adoption[col] = pct

    plt.figure(figsize=(10, 4))
    colors = ['#e74c3c' if v < 50 else '#2ecc71' for v in adoption.values()]
    # Use pretty labels for y-axis
    y_labels = [risk_labels.get(col, col) for col in adoption.keys()]
    sns.barplot(x=list(adoption.values()), y=y_labels, palette=colors)
    plt.title("Security & Compliance Adoption Rates (%)")
    plt.xlim(0, 100)
    plt.axvline(50, color='gray', linestyle='--')
    return plot_to_base64(plt.gcf())

def create_tool_stack(df):
    """ERP Market Share Pie Chart"""
    plt.figure(figsize=(6, 6))
    counts = df['erp_in_use'].value_counts().head(6) # Top 6 only
    plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
    plt.title("ERP Market Share")
    return plot_to_base64(plt.gcf())


HTML_REPORT_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Informe de Mercado de Consultoría</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 0; color: #333; background: #f4f7f6; }
        .container { width: 850px; margin: 0 auto; background: white; padding: 40px; box-shadow: 0 0 10px rgba(0,0,0,0.1); }
        
        /* HEADER */
        header { border-bottom: 2px solid #003366; padding-bottom: 20px; margin-bottom: 40px; }
        h1 { color: #003366; font-size: 28px; margin: 0; }
        .subtitle { color: #666; font-size: 14px; margin-top: 5px; }
        
        /* KPIS */
        .kpi-row { display: flex; justify-content: space-between; margin-bottom: 40px; }
        .kpi-card { background: #eef2f5; padding: 20px; border-radius: 8px; width: 30%; text-align: center; border-top: 4px solid #003366; }
        .kpi-value { font-size: 32px; font-weight: bold; color: #003366; display: block; }
        .kpi-label { font-size: 12px; text-transform: uppercase; letter-spacing: 1px; color: #555; }
        
        /* SECTIONS */
        .section { margin-bottom: 50px; page-break-inside: avoid; }
        h2 { color: #003366; border-left: 5px solid #2ecc71; padding-left: 15px; margin-bottom: 20px; }
        p { line-height: 1.6; color: #555; }
        
        /* CHARTS */
        .chart-box { text-align: center; margin: 20px 0; border: 1px solid #eee; padding: 10px; border-radius: 5px; }
        img { max-width: 100%; height: auto; }
        
        /* FOOTER */
        .footer { text-align: center; font-size: 11px; color: #999; margin-top: 50px; border-top: 1px solid #eee; padding-top: 10px; }
    </style>
</head>
<body>

<div class="container">

    <header>
        <h1>Análisis Digital del Mercado 2024</h1>
        <div class="subtitle">Generado para Revisión Interna de Consultoría | {{ date }}</div>
    </header>

    <div class="kpi-row">
        <div class="kpi-card">
            <span class="kpi-value">{{ total_companies }}</span>
            <span class="kpi-label">Empresas Analizadas</span>
        </div>
        <div class="kpi-card">
            <span class="kpi-value">{{ avg_score }}</span>
            <span class="kpi-label">Punt. Madurez Promedio</span>
        </div>
        <div class="kpi-card">
            <span class="kpi-value">{{ risk_pct }}%</span>
            <span class="kpi-label">Clientes de Alto Riesgo</span>
        </div>
    </div>

    <div class="section">
        <h2>1. Visión General de Madurez de Mercado</h2>
        <p>
            El mercado muestra una distribución en forma de campana, con la mayoría de empresas en la etapa de "En Desarrollo".
            Se observa una clara división entre líderes y rezagados digitales.
        </p>
        <div class="chart-box">
            <img src="data:image/png;base64,{{ chart_dist }}">
        </div>
    </div>

    <div class="section">
        <h2>2. Desempeño por Sector</h2>
        <p>
            Las empresas <strong>Industriales</strong> muestran consistencia pero menor techo de innovación.
            <strong>Servicios</strong> presenta la mayor variabilidad, indicando un mercado fragmentado y con potencial de consolidación.
        </p>
        <div class="chart-box">
            <img src="data:image/png;base64,{{ chart_sector }}">
        </div>
    </div>

    <div class="section">
        <h2>3. Análisis de Seguridad y Riesgo</h2>
        <p>
            <strong style="color: #e74c3c;">Hallazgo Crítico:</strong> Factores básicos como 2FA y copias de seguridad están
            muy poco adoptados en el segmento Micro/Pequeña. Representa la principal oportunidad comercial inmediata.
        </p>
        <div class="chart-box">
            <img src="data:image/png;base64,{{ chart_risk }}">
        </div>
    </div>

    <div class="section">
        <h2>4. El Stack Tecnológico</h2>
        <p>
            Análisis de cuota de mercado de los principales ERP. "Ninguno" representa la oportunidad Greenfield.
        </p>
        <div class="chart-box">
            <img src="data:image/png;base64,{{ chart_stack }}">
        </div>
    </div>

    <div class="footer">
        Informe automatizado generado por el Motor de Análisis Python. Fuente de datos: Encuestas internas CSV.
    </div>

</div>

</body>
</html>
"""

# ==========================================
# 3. GENERATOR FUNCTION
# ==========================================

def generate_pdf_report(df):
    print("Generating Narrative Report...")
    
    # 1. Calc KPIs
    # Define "High Risk" as Security Score < 40
    risk_pct = round((len(df[df['KPI_SECURITY'] < 40]) / len(df)) * 100, 1)
    
    # 2. Generate Plots
    chart_dist = create_executive_score_dist(df)
    chart_sector = create_sector_comparison(df)
    chart_risk = create_risk_matrix(df)
    chart_stack = create_tool_stack(df)
    
    # 3. Render HTML
    template = Template(HTML_REPORT_TEMPLATE)
    html_content = template.render(
        date=datetime.date.today().strftime("%d %b %Y"),
        total_companies=len(df),
        avg_score=round(df['GLOBAL_SCORE'].mean(), 1),
        risk_pct=risk_pct,
        chart_dist=chart_dist,
        chart_sector=chart_sector,
        chart_risk=chart_risk,
        chart_stack=chart_stack
    )
    
    # 4. Save
    with open("Market_Analysis_Report.html", "w", encoding="utf-8") as f:
        f.write(html_content)
        
    print("Report 'Market_Analysis_Report.html' created.")
    print("Double-click to open. Print -> Save as PDF to finalize.")

# --- USAGE ---
generate_pdf_report(master_df)

Generating Narrative Report...
Report 'Market_Analysis_Report.html' created.
Double-click to open. Print -> Save as PDF to finalize.


/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_93991/4051938576.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='GLOBAL_SCORE', y='company_profile_cnae', data=df, order=order, palette="Blues_r")
/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_93991/4051938576.py:63: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=list(adoption.values()), y=y_labels, palette=colors)
